# BBC News Classification with Cohere Embeddings

**Workflow**
1. Load BBC dataset (5 classes)
2. Take 2 paragraphs from each class as training set (10 docs)
3. Embed test docs via Cohere in batches of 100 with 60s wait between batches (save each batch)
4. Train LogisticRegression on training embeddings, predict on test
5. Generate classification report

**Setup**: get a free trial key from https://dashboard.cohere.com → set `COHERE_API_KEY` below.

## Setup
Install deps, import libraries, configure key & paths.

In [2]:
# Install once
# !pip install cohere pandas scikit-learn numpy
# %pip install cohere

In [3]:
import os, time, json, pickle
import numpy as np
import pandas as pd
import cohere
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# --- CONFIG ---
COHERE_API_KEY = os.getenv("COHERE_API_KEY", "9md32CI661W4YST1Gr5deYyMNEHRHHPhb7U7SDrU")
DATA_PATH      = "bbc_news_full.csv"   # put the csv next to this notebook
EMBED_MODEL    = "embed-english-v3.0"
BATCH_SIZE     = 100
WAIT_SECONDS   = 60                    # stay under trial rate limit
TEST_SIZE      = 200                   # small run
RANDOM_STATE   = 0
BATCH_DIR      = "embed_batches"       # per-batch results saved here

os.makedirs(BATCH_DIR, exist_ok=True)
co = cohere.Client(COHERE_API_KEY)

In [4]:
# from google.colab import files
# uploaded = files.upload()  # drag & drop bbc_news_full.csv

## 1. Load data — 2 docs/class as training, stratified test sample

In [5]:
df = pd.read_csv(DATA_PATH)
print("shape:", df.shape)
print(df['label_text'].value_counts())

# 2 per class = training
train_df = (df.groupby('label_text', group_keys=False)
              .apply(lambda g: g.sample(2, random_state=RANDOM_STATE)))

# remaining pool, then take a stratified sample of TEST_SIZE
pool = df.drop(train_df.index)
test_df, _ = train_test_split(
    pool, train_size=TEST_SIZE,
    stratify=pool['label_text'], random_state=RANDOM_STATE)

train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)
print("train:", train_df.shape, " test:", test_df.shape)
train_df

shape: (2225, 3)
label_text
sport            511
business         510
politics         417
tech             401
entertainment    386
Name: count, dtype: int64
train: (10, 3)  test: (200, 3)


C:\Users\hp\AppData\Local\Temp\ipykernel_11500\1701377244.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(2, random_state=RANDOM_STATE)))


,text,label,label_text
0,saudi investor picks up the savoy london s fam...,1,business
1,mg rover china tie-up delayed mg rover s pro...,1,business
2,little britain vies for tv trophy bbc hits lit...,3,entertainment
3,hundreds vie for best film oscar a total of 26...,3,entertainment
4,manchester wins labour conference the labour p...,4,politics
5,jowell rejects las vegas jibe the secretary ...,4,politics
6,liverpool pledge to keep gerrard liverpool chi...,2,sport
7,italy 8-38 wales wales secured their first awa...,2,sport
8,internet boom for gift shopping cyberspace is ...,0,tech
9,blog reading explodes in america americans are...,0,tech


In [ ]:
test_df['label_text'].value_counts()

## 2. Embed in batches of 100 (60s waits, saved per batch)

In [ ]:
def embed_batch(texts, input_type):
    """Call Cohere embed for one batch."""
    resp = co.embed(texts=texts, model=EMBED_MODEL, input_type=input_type)
    return np.array(resp.embeddings)

def embed_in_batches(texts, input_type, tag):
    all_vecs = []
    n = len(texts)
    for i in range(0, n, BATCH_SIZE):
        batch = texts[i : i + BATCH_SIZE]
        batch_idx = i // BATCH_SIZE
        print(f"[{tag}] batch {batch_idx+1} | docs {i}..{i+len(batch)-1}")

        vecs = embed_batch(batch, input_type)
        all_vecs.append(vecs)

        # save this batch to disk
        out = os.path.join(BATCH_DIR, f"{tag}_batch_{batch_idx:03d}.npy")
        np.save(out, vecs)
        print(f"  saved {out}  shape={vecs.shape}")

        # wait before next batch (skip after the last one)
        if i + BATCH_SIZE < n:
            print(f"  sleeping {WAIT_SECONDS}s...")
            time.sleep(WAIT_SECONDS)

    return np.vstack(all_vecs)

# training texts — tiny, one batch
X_train = embed_in_batches(train_df['text'].tolist(),
                           input_type='search_document', tag='train')
y_train = train_df['label_text'].values

# test texts — batches of 100 with 60s waits
X_test  = embed_in_batches(test_df['text'].tolist(),
                           input_type='search_document', tag='test')
y_test  = test_df['label_text'].values

print("X_train:", X_train.shape, " X_test:", X_test.shape)

## Baseline — LogisticRegression on Cohere embeddings
This is the reference accuracy that the four methods below are compared against.

In [ ]:
clf = LogisticRegression(max_iter=1000, C=1.0)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

# save predictions
out_df = test_df.copy()
out_df['predicted'] = y_pred
out_df.to_csv('predictions.csv', index=False)
print('saved predictions.csv')

In [ ]:
print(classification_report(y_test, y_pred, digits=3))

labels = sorted(df['label_text'].unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
print('Confusion matrix (rows=true, cols=pred)')
pd.DataFrame(cm, index=labels, columns=labels)

### (Optional) Learning curve for the baseline

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LearningCurveDisplay, StratifiedKFold
import matplotlib.pyplot as plt
import numpy as np


cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=0)

LearningCurveDisplay.from_estimator(
    clf,
    X_train,
    y_train,
    cv=cv,
    scoring="accuracy",
    train_sizes=np.linspace(0.4, 1.0, 4),
    n_jobs=-1
)

plt.title("Learning Curve - Logistic Regression")
plt.show()

## Method comparison

The four cells below re-use `X_train`, `X_test`, `y_train`, `y_test`, `train_df`, `test_df`,
and the `embed_batch` helper from above. Each method appends its accuracy to the
`results` dict; the final cell renders a sorted summary table.

In [ ]:
from sklearn.metrics import accuracy_score
results = {'LogisticRegression (baseline)': accuracy_score(y_test, y_pred)}
print(results)

## Method 1 — Embed + alternative classifier heads

Cohere embeddings, different classifiers. No extra API calls.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

heads = {
    'kNN (k=3, cosine)': KNeighborsClassifier(n_neighbors=3, metric='cosine'),
    'SVM (linear)'     : SVC(kernel='linear', C=1.0),
    'RandomForest'     : RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
}

for name, model in heads.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    acc  = accuracy_score(y_test, pred)
    results[name] = acc
    print(f'--- {name}  acc={acc:.3f} ---')
    print(classification_report(y_test, pred, digits=3))

## Method 2 — Embed + semantic similarity (zero training)

Embed a short description of each class, then assign each test doc to the class whose
embedding is most similar (cosine). No classifier is trained at all.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

class_descriptions = {
    'business'     : 'business, finance, economy, companies, markets, and corporate news',
    'entertainment': 'entertainment, movies, music, celebrities, film, and television',
    'politics'     : 'politics, government, elections, policy, and political parties',
    'sport'        : 'sports, football, cricket, rugby, tennis, athletes, and matches',
    'tech'         : 'technology, computers, software, internet, gadgets, and science',
}
class_labels = list(class_descriptions.keys())
class_texts  = list(class_descriptions.values())

# one tiny embed call for the 5 class descriptions
class_vecs = embed_batch(class_texts, input_type='search_query')

sims      = cosine_similarity(X_test, class_vecs)
pred_sim  = np.array([class_labels[i] for i in sims.argmax(axis=1)])
acc_sim   = accuracy_score(y_test, pred_sim)
results['Semantic similarity (zero-shot)'] = acc_sim
print(f'acc = {acc_sim:.3f}')
print(classification_report(y_test, pred_sim, digits=3))

## Method 3 — Command chat prompting (few-shot)

For each test doc we ask the Command model to pick a class, using the 2-per-class training
docs as in-context examples. We batch calls with the same 60s wait pattern.

Note: this sends **one API call per test doc**, so it is slower and costs more than embed-based methods.

In [ ]:
CHAT_MODEL = 'command-a-03-2025'   # or 'command-r' for cheaper/faster
CHAT_BATCH = 20                 # requests before each sleep (chat rate limits are tighter)

labels_sorted = sorted(df['label_text'].unique())
preamble = (
    'You are a news classifier. Reply with ONLY one of these labels: '
    + ', '.join(labels_sorted) + '. No other words, no punctuation.'
)

# build few-shot examples from train_df
examples = []
for _, row in train_df.iterrows():
    examples.append(f'TEXT: {row["text"][:600]}\nLABEL: {row["label_text"]}')
few_shot = '\n\n'.join(examples)

def classify_one(text):
    prompt = (few_shot
              + f'\n\nTEXT: {text[:1500]}\nLABEL:')
    resp = co.chat(model=CHAT_MODEL, message=prompt, preamble=preamble, temperature=0)
    out  = resp.text.strip().lower().split()[0]
    # snap to nearest valid label
    return out if out in labels_sorted else min(labels_sorted,
                                                 key=lambda l: 0 if l in out else 1)

preds_chat = []
for i, text in enumerate(test_df['text'].tolist()):
    preds_chat.append(classify_one(text))
    if (i + 1) % 20 == 0:
        print(f'  {i+1}/{len(test_df)} done')
    if (i + 1) % CHAT_BATCH == 0 and (i + 1) < len(test_df):
        print(f'  sleeping {WAIT_SECONDS}s...')
        time.sleep(WAIT_SECONDS)

preds_chat = np.array(preds_chat)
acc_chat = accuracy_score(y_test, preds_chat)
results['Command chat (few-shot)'] = acc_chat
print(f'acc = {acc_chat:.3f}')
print(classification_report(y_test, preds_chat, digits=3))

## Method 4 — Rerank as zero-shot classifier

For each test doc, treat every class description as a candidate "document" and the test
text as the "query." The highest rerank score wins. One API call per test doc.

In [ ]:
# RERANK_MODEL = 'rerank-english-v3.0'
RERANK_MODEL = 'rerank-v3.5'
RERANK_BATCH = 8

rerank_docs = [
    'This is a news article about business, finance, the economy, companies, and markets.',
    'This is a news article about entertainment, movies, music, celebrities, and television.',
    'This is a news article about politics, government, elections, and political parties.',
    'This is a news article about sports, athletes, matches, football, cricket, or rugby.',
    'This is a news article about technology, computers, software, internet, and gadgets.',
]
# class_labels stays the same order: business, entertainment, politics, sport, tech

preds_rr = []
for i, text in enumerate(test_df['text'].tolist()):
    r = co.rerank(model=RERANK_MODEL,
                  query=text[:400],
                  documents=rerank_docs,
                  top_n=1)
    preds_rr.append(class_labels[r.results[0].index])
    if (i + 1) % 20 == 0:
        print(f'  {i+1}/{len(test_df)} done')
    if (i + 1) % RERANK_BATCH == 0 and (i + 1) < len(test_df):
        print(f'  sleeping {WAIT_SECONDS}s...')
        time.sleep(WAIT_SECONDS)

preds_rr = np.array(preds_rr)
acc_rr   = accuracy_score(y_test, preds_rr)
results['Rerank (zero-shot)'] = acc_rr
print(f'acc = {acc_rr:.3f}')
print(classification_report(y_test, preds_rr, digits=3))

## Summary — all methods

In [ ]:
summary = pd.DataFrame(
    [(name, f'{acc:.3f}') for name, acc in results.items()],
    columns=['method', 'accuracy']
).sort_values('accuracy', ascending=False).reset_index(drop=True)
summary.to_csv('method_comparison.csv', index=False)
summary

### Notes
- Trial keys are free but rate-limited (~100 embed calls/min). The 60s wait between batches of 100 keeps you safely under that.
- Only 10 training docs is tiny — expect accuracy in the 70-85% range. For stronger results, bump training to 20-50 per class.
- Per-batch `.npy` files are in `embed_batches/` — safe to resume from if interrupted.